In [1]:
# ChakraModel R6 — negative-source diversity. Paste this whole file into ONE Kaggle code cell.
# Settings: GPU T4 x2, Internet ON. Inputs: chakra-leakbench-v1, chakra-leakbench-e4-v1,
# chakra-leakbench-s2-v1, chakra-leakbench-r2-v1-22-09-14-50. Then Save Version -> Save & Run All.
import subprocess
subprocess.run('pip -q install segmentation-models-pytorch==0.5.0 timm', shell=True)

# R6: does negative-sample DIVERSITY fix cross-centre false alarms?
# E4 trained on HyperKvasir negatives only and halved false alarms at home (85%->3%) but only partly
# elsewhere (84%->27%). Here the SAME budget of polyp-free frames comes from one source or two:
#   pgneg   : PolypGen negatives only        (from 11 training sequences)
#   mixneg  : half HyperKvasir, half PolypGen
# (the HyperKvasir-only arm is E4, already trained). Held-out: 12 PolypGen sequences never trained on,
# the HyperKvasir test pool, plus PICCOLO and PolypGen single-frame negatives as a third source.
# Inputs: chakra-leakbench-v1, chakra-leakbench-e4-v1, chakra-leakbench-s2-v1, chakra-leakbench-r2-...
# HALF = "" runs all 12 (~2.5 h on 2 GPUs). "A" = first 6, "B" = last 6.
HALF = ""
import glob, os, csv, random, collections, subprocess, sys
IN = "/kaggle/input"
def find_dir(marker, depth=6):
    for d in range(1, depth + 1):
        h = glob.glob(os.path.join(IN, *(["*"] * d), marker))
        if h: return os.path.dirname(h[0])
    raise FileNotFoundError(marker)
S1 = find_dir("fixed_polyp_dataset_v1"); E4 = find_dir("negatives_hyperkvasir_v1")
S2 = find_dir("piccolo_clean"); R2 = find_dir("runs_e1")
PG = find_dir("negatives_polypgen_v1")
ROOT = "/kaggle/working/root"; W = "/kaggle/working/leakbench"; os.makedirs(ROOT, exist_ok=True)
print("S1", S1, "\nE4", E4, "\nS2", S2, "\nPG", PG, "\nCODE", R2, flush=True)
for src in (S1, E4, S2):
    for e in os.listdir(src):
        if e not in ("leakbench", "runs_e1") and not os.path.exists(f"{ROOT}/{e}"): os.symlink(f"{src}/{e}", f"{ROOT}/{e}")
subprocess.run(f"rm -rf {W} && cp -r {R2}/leakbench {W}", shell=True, check=True)

rd = lambda p: list(csv.DictReader(open(p)))
v2 = rd(f"{W}/manifests/v2.csv"); base = [r for r in v2 if r["split"] in ("train", "val")]
tests = [r for r in rd(f"{W}/manifests/r2_e4_eval.csv") if r["split"].startswith("test_")]
hk = [r for r in rd(f"{W}/manifests/r2_e4_eval.csv") if r["split"] == "neg_hyperkvasir"]      # HK held-out pool
hkp = f"{W}/manifests/e4_negtrain.csv"
if os.path.exists(hkp):
    hk_train = [r for r in rd(hkp) if r["split"] == "train" and "hyperkvasir" in r["image"]]
else:   # rebuild E4's HyperKvasir training pool from the negatives manifest (role=train_pool)
    hk_train = [dict(image="negatives_hyperkvasir_v1/" + r["image"], mask="", split="train",
                     source="hyperkvasir_neg|seen", group=r["group"])
                for r in rd(f"{E4}/negatives_hyperkvasir_v1/manifest.csv") if r["role"] == "train_pool"]
pg = [r for r in rd(f"{W}/manifests/r2_e4_eval.csv") if r["split"] == "neg_polypgen"]
s2n = [r for r in rd(f"{W}/manifests/r2_s2_eval.csv") if r["split"] in ("neg_piccolo", "neg_polypgen_single")]
seq = lambda r: r["image"].split("/")[-1].split("_neg__")[0]
SEQS = sorted({seq(r) for r in pg}); random.Random(7).shuffle(SEQS)
TR_SEQ, TE_SEQ = set(SEQS[:11]), set(SEQS[11:])
pg_pool = [r for r in pg if seq(r) in TR_SEQ]; pg_held = [r for r in pg if seq(r) in TE_SEQ]
N = len(hk_train)                                   # same negative budget as E4 (20% of training)
rng = random.Random(11)
row = lambda r, s: dict(image=r["image"], mask="", split=s, source=r.get("source", "neg"), group=r.get("group", ""))
mk = lambda rows: [row(r, "train") for r in rows]
arms = {"pgneg":  mk(rng.sample(pg_pool, N)),
        "mixneg": mk(rng.sample([r for r in hk_train], N // 2) + rng.sample(pg_pool, N - N // 2))}
ev = tests + [row(r, "neg_pg_heldout") for r in pg_held] + [row(r, "neg_hyperkvasir") for r in hk] \
     + [row(r, "neg_thirdsource") for r in s2n]
for name, neg in arms.items():
    rows = base + neg + ev
    with open(f"{W}/manifests/r6_{name}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["image", "mask", "split", "source", "group"], extrasaction="ignore")
        w.writeheader(); w.writerows(rows)
    print(f"r6_{name}.csv", dict(collections.Counter(r["split"] for r in rows)), flush=True)
print("PolypGen sequences: train", len(TR_SEQ), "held out", len(TE_SEQ), "| negative budget per arm", N)
jobs = [dict(job=200 + i, experiment="R6_negdiversity", manifest=f"manifests/r6_{arm}.csv", model=m, seed=s, epochs=50)
        for i, (s, arm, m) in enumerate((s, arm, m) for s in (42, 43, 44) for arm in ("pgneg", "mixneg")
                                        for m in ("xattn", "unet_r34"))]
with open(f"{W}/jobs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(jobs[0])); w.writeheader(); w.writerows(jobs)
print("jobs", len(jobs), jobs[0]["job"], "-", jobs[-1]["job"], flush=True)
if os.environ.get("DRY") == "1": sys.exit(0)
SL = {"": ("200-205", "206-211"), "A": ("200-202", "203-205"), "B": ("206-208", "209-211")}[HALF]
cmd = lambda g: (f"CUDA_VISIBLE_DEVICES={g} python {W}/run_queue.py --data-root {ROOT} --jobs {SL[g]} "
                 f"--out /kaggle/working/runs_r6 --budget-hours 11.0 >> /kaggle/working/log_r6_gpu{g}.txt 2>&1")
procs = [subprocess.Popen(cmd(g), shell=True) for g in (0, 1)]; print([p.wait() for p in procs])
print("finished runs", len(glob.glob("/kaggle/working/runs_r6/*/results.json")))
subprocess.run(f"python {W}/aggregate.py /kaggle/working/runs_r6", shell=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.6 MB/s eta 0:00:00
S1 /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1 
E4 /kaggle/input/datasets/gokulrocky/chakra-leakbench-e4-v1 
S2 /kaggle/input/datasets/gokulrocky/chakra-leakbench-s2-v1 
PG /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1 
CODE /kaggle/input/datasets/gokulrocky/chakra-leakbench-r2-v1-22-09-14-50
r6_pgneg.csv {'train': 1536, 'val': 222, 'test_kvasir': 100, 'test_clinicdb': 62, 'test_colondb': 380, 'test_etis': 196, 'test_cvc300': 60, 'neg_pg_heldout': 2404, 'neg_hyperkvasir': 2947, 'neg_thirdsource': 168}
r6_mixneg.csv {'train': 1536, 'val': 222, 'test_kvasir': 100, 'test_clinicdb': 62, 'test_colondb': 380, 'test_etis': 196, 'test_cvc300': 60, 'neg_pg_heldout': 2404, 'neg_hyperkvasir': 2947, 'neg_thirdsource': 168}
PolypGen sequences: train 11 held out 12 | negative budget per arm 308
jobs 12 200 - 211
[0, 0]
finished runs 12
r6_mixneg.csv unet_r34       n=3 val=0.885±0.002 kvasir=0.908±0.002 

CompletedProcess(args='python /kaggle/working/leakbench/aggregate.py /kaggle/working/runs_r6', returncode=0)